# Image Browser

Change the filters in the second cell and rerun to browse.

In [ ]:
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path.cwd().parent

train = pd.read_csv(PROJECT_ROOT / "data/processed/train_annotations.csv")
test = pd.read_csv(PROJECT_ROOT / "data/processed/test_annotations.csv")

for df in (train, test):
    df["resolution"] = df["width"].astype(str) + "x" + df["height"].astype(str)

CLASSES = sorted(train["category_name"].dropna().unique())
CMAP = plt.colormaps["tab20"].resampled(len(CLASSES))
CLASS_COLOURS = {name: CMAP(i) for i, name in enumerate(CLASSES)}


def draw_bboxes(ax, rows):
    for _, r in rows.iterrows():
        if pd.isna(r.get("category_name")):
            continue
        c = CLASS_COLOURS[r["category_name"]]
        ax.add_patch(patches.Rectangle(
            (r["bbox_x"], r["bbox_y"]), r["bbox_w"], r["bbox_h"],
            linewidth=2, edgecolor=c, facecolor="none",
        ))
        ax.text(
            r["bbox_x"], r["bbox_y"] - 4, r["category_name"],
            color=c, fontsize=8, fontweight="bold",
            bbox=dict(facecolor="black", alpha=0.6, pad=1, edgecolor="none"),
        )


print(f"Classes: {CLASSES}")
print(f"Resolutions (train): {sorted(train['resolution'].unique())}")

In [ ]:
# --- Change these and rerun ---
SPLIT = "train"              # "train" or "test"
CLASS_FILTER = "All"         # "All" or a species name
RESOLUTION = "All"           # "All" or e.g. "1920x1080"
MULTI_EGG_ONLY = False       # True = only images with 2+ annotations
START = 0
N = 5

full = train if SPLIT == "train" else test
filtered = full
if CLASS_FILTER != "All":
    filtered = filtered[filtered["category_name"] == CLASS_FILTER]
if RESOLUTION != "All":
    filtered = filtered[filtered["resolution"] == RESOLUTION]
if MULTI_EGG_ONLY:
    counts = filtered.groupby("image_id")["annotation_id"].count()
    filtered = filtered[filtered["image_id"].isin(counts[counts > 1].index)]

ids = sorted(filtered["image_id"].unique())
end = min(START + N, len(ids))
print(f"Images {START + 1}–{end} of {len(ids)}")

for img_id in ids[START:end]:
    rows = full[full["image_id"] == img_id]
    r = rows.iloc[0]
    species = sorted(rows["category_name"].dropna().unique())

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(Image.open(PROJECT_ROOT / r["file_path"]))
    draw_bboxes(ax, rows)
    ax.set_title(f"{r['file_name']}  |  {r['resolution']}  |  {', '.join(species)}", fontsize=10)
    ax.axis("off")
    plt.tight_layout()
    plt.show()